# Medicare+ — Prescription Handwriting OCR
## Notebook 2 of 3: Fine-tune TrOCR on Doctor Handwriting

**Goal:** Fine-tune [`microsoft/trocr-base-handwritten`](https://huggingface.co/microsoft/trocr-base-handwritten) on the prescription dataset prepared in notebook 1, then save the model to Google Drive and offer a zip download.

**Prerequisite:** You ran `01_data_prep.ipynb` and have the prepared dataset at `/content/drive/MyDrive/medicare_plus_ocr/data/prepared`.

**Runtime:** Colab GPU (T4) is required. Expect ~1–3 hours for 10 epochs on ~4,000 training images. Adjust `NUM_EPOCHS` if you need faster turnaround.

### Outline
1. Install dependencies
2. Mount Drive, load prepared dataset
3. Load TrOCR processor + model
4. Define training transforms & data collator
5. Configure HuggingFace `Seq2SeqTrainer`
6. Train with periodic CER evaluation
7. Save final model to Drive + zip for download

## 1. Install dependencies

Pin to known-good versions to avoid surprise API changes.

In [ ]:
!pip install -q \
    'transformers>=4.44,<5' \
    'datasets>=2.20' \
    'accelerate>=0.33' \
    'evaluate>=0.4' \
    'jiwer>=3' \
    sentencepiece pillow

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

## 2. Mount Drive & load prepared dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/medicare_plus_ocr'
PREPARED_DIR = f'{PROJECT_ROOT}/data/prepared'
MODEL_OUT_DIR = f'{PROJECT_ROOT}/models/trocr-prescription'
LOG_DIR = f'{PROJECT_ROOT}/logs'
os.makedirs(MODEL_OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

from datasets import load_from_disk
splits = load_from_disk(PREPARED_DIR)
splits

## 3. Load TrOCR processor + model

`microsoft/trocr-base-handwritten` was pretrained on the IAM handwriting dataset — a good starting point because it already handles cursive English. We replace the decoder head only minimally; fine-tuning continues end-to-end.

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

BASE_MODEL = 'microsoft/trocr-base-handwritten'
MAX_TARGET_LENGTH = 32

processor = TrOCRProcessor.from_pretrained(BASE_MODEL)
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.vocab_size = model.config.decoder.vocab_size

model.config.max_length = MAX_TARGET_LENGTH
model.config.num_beams = 4
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0

## 4. Dataset wrapper

TrOCR expects `pixel_values` and `labels`. The processor handles both image preprocessing and tokenization.

In [ ]:
from torch.utils.data import Dataset

class TrOCRDataset(Dataset):
    def __init__(self, hf_split, processor, max_target_length=MAX_TARGET_LENGTH):
        self.split = hf_split
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.split)

    def __getitem__(self, idx):
        row = self.split[idx]
        image = row['image'].convert('RGB')
        text = row['label']
        pixel_values = self.processor(image, return_tensors='pt').pixel_values[0]
        labels = self.processor.tokenizer(
            text,
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True,
        ).input_ids
        labels = [t if t != self.processor.tokenizer.pad_token_id else -100 for t in labels]
        return {'pixel_values': pixel_values, 'labels': torch.tensor(labels)}

train_ds = TrOCRDataset(splits['train'], processor)
val_ds = TrOCRDataset(splits['validation'], processor)
print(f'train samples: {len(train_ds)}')
print(f'val   samples: {len(val_ds)}')

## 5. CER metric for evaluation

In [ ]:
import evaluate

cer_metric = evaluate.load('cer')

def compute_metrics(eval_pred):
    pred_ids, label_ids = eval_pred
    label_ids_clean = [
        [t for t in seq if t != -100] for seq in label_ids
    ]
    label_str = processor.batch_decode(label_ids_clean, skip_special_tokens=True)
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {'cer': cer}

## 6. Configure training

Tune these for your situation:
* `NUM_EPOCHS=10` is a sensible starting point. More if your CER is still dropping at the end.
* `LEARNING_RATE=5e-5` is the standard TrOCR fine-tuning rate.
* Reduce `BATCH_SIZE` if you hit OOM on T4.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator

NUM_EPOCHS = 10
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 5e-5

training_args = Seq2SeqTrainingArguments(
    output_dir=f'{LOG_DIR}/checkpoints',
    overwrite_output_dir=True,
    predict_with_generate=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    fp16=torch.cuda.is_available(),
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    warmup_steps=200,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics,
    data_collator=default_data_collator,
)

## 7. Train

This is the long step. With ~4,000 training images at batch size 8 and 10 epochs on a T4, expect roughly 1–3 hours.

In [ ]:
trainer.train()

## 8. Save final model to Drive

In [ ]:
trainer.save_model(MODEL_OUT_DIR)
processor.save_pretrained(MODEL_OUT_DIR)
print(f'Model saved to: {MODEL_OUT_DIR}')
!ls -lah {MODEL_OUT_DIR}

## 9. Zip the model and offer a download link

Drop the resulting zip into your repo under `server/models/` (gitignored) when you want to swap from the base pretrained model to your fine-tuned one. Set `PRESCRIPTION_MODEL_PATH=server/models/trocr-prescription` to activate it in the backend.

In [ ]:
import shutil
from google.colab import files

ZIP_PATH = '/content/trocr-prescription.zip'
shutil.make_archive(ZIP_PATH.replace('.zip', ''), 'zip', MODEL_OUT_DIR)
print(f'Zip size: {os.path.getsize(ZIP_PATH) / 1e6:.1f} MB')
files.download(ZIP_PATH)

## Done

Move on to **`03_evaluate.ipynb`** for proper test-set evaluation and qualitative analysis.